## THIRUMALAI K
## 212224240176

In [27]:
%pip install groq

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
import math
import json
from groq import Groq

In [29]:
GROQ_API_KEY = "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"

In [30]:
client = Groq(api_key=GROQ_API_KEY)
print("Groq Client initialized successfully.")

Groq Client initialized successfully.


In [31]:
models = client.models.list()
for m in models.data:
  print(m.id)

openai/gpt-oss-20b
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
allam-2-7b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
openai/gpt-oss-120b
whisper-large-v3-turbo


In [32]:
def calculate_cylinder_volume(radius: float, height: float, unit: str = "cm") -> str:
    """Calculates the volume of a cylinder using formula: V = π * r² * h"""
    try:
        r = float(radius)
        h = float(height)
        if r <= 0 or h <= 0:
            return json.dumps({"error": "Radius and height must be positive numbers."})
            
        volume = math.pi * (r ** 2) * h
        
        result = {
            "radius": r,
            "height": h,
            "unit": unit,
            "volume": round(volume, 4),
            "volume_unit": f"cubic {unit} ({unit}³)",
            "formula_applied": "V = π * r^2 * h"
        }
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"error": str(e)})

In [33]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_cylinder_volume",
            "description": "Calculates the volume of a cylinder using its radius and height.",
            "parameters": {
                "type": "object",
                "properties": {
                    "radius": {
                        "type": "number",
                        "description": "Radius of the base circle."
                    },
                    "height": {
                        "type": "number",
                        "description": "Vertical height of the cylinder."
                    },
                    "unit": {
                        "type": "string",
                        "description": "Measurement unit (e.g., cm, m, inches)"
                    }
                },
                "required": ["radius", "height"]
            }
        }
    }
]

In [34]:
print("Tool schema defined successfully.")

Tool schema defined successfully.


In [35]:
messages = [
    {
        "role": "system",
        "content": "You are a precise mathematical assistant. Always use the calculate_cylinder_volume tool to calculate the volume of a cylinder."
    },
    {
        "role": "user",
        "content": "What is the volume of a cylinder with radius 5 cm and height 12 cm?"
    }
]

In [36]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

In [37]:
response_message = response.choices[0].message
print("Model Requested Tool Call:")
print(response_message.tool_calls)

Model Requested Tool Call:
[ChatCompletionMessageToolCall(id='fc_57ec3e1a-9e65-463f-8dc2-cd0784662eb7', function=Function(arguments='{"height":12,"radius":5,"unit":"cm"}', name='calculate_cylinder_volume'), type='function')]


In [38]:
if response_message.tool_calls:
    
    messages.append(response_message)
    
    
    for tool_call in response_message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        print(f"\nExecuting '{func_name}' locally with arguments:", func_args)
        
        if func_name == "calculate_cylinder_volume":
            tool_output = calculate_cylinder_volume(
                radius=func_args.get("radius"),
                height=func_args.get("height"),
                unit=func_args.get("unit", "cm")
            )
            
            # 3. Append the execution output back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_output
            })


Executing 'calculate_cylinder_volume' locally with arguments: {'height': 12, 'radius': 5, 'unit': 'cm'}


In [39]:
final_response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
    )

In [40]:
print("THIRUMALAI K")
print("212224240176")
import re
from IPython.display import Markdown, display

content = final_response.choices[0].message.content

# 1. Convert display math \[ ... \] or [ ... \pi ... ] to $$ ... $$
content = re.sub(
    r"\\?\[\s*(.*?\\pi.*?)\s*\\?\]", r"$$\1$$", content, flags=re.DOTALL
)

# 2. Convert inline \( ... \) to $ ... $
content = re.sub(r"\\?\(\s*([Vv])\s*\\?\)", r"$\1$", content)

# 3. Clean any leftover literal escaped brackets around math
content = content.replace(r"\[", "$$").replace(r"\]", "$$")
content = content.replace(r"\(", "$").replace(r"\)", "$")

print("\n" + "=" * 50)
print("FINAL LAB REPORT OUTPUT:")
print("=" * 50)
display(Markdown(content))

THIRUMALAI K
212224240176

FINAL LAB REPORT OUTPUT:


The volume $V$ of a cylinder is given by  

$$V = \pi r^{2} h .$$

Plugging in the values $r = 5\text{ cm}$ and $h = 12\text{ cm}$:

$$V = \pi \times (5\text{ cm})^{2} \times 12\text{ cm}
   = \pi \times 25\text{ cm}^{2} \times 12\text{ cm}
   = \pi \times 300\text{ cm}^{3}
   \approx 942.48\text{ cm}^{3}.$$

**Volume:** **≈ 942.48 cubic centimeters (cm³).**